Exploring the dataset

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, balanced_accuracy_score, f1_score, make_scorer
from sklearn.model_selection import train_test_split, PredefinedSplit, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import optuna
from optuna.samplers import TPESampler

## Load Data

In [ ]:
# simplified working path (notebook runs from `notebooks/`)
data_path = Path('..') / 'data' / 'BTCUSDT' / 'processed' / 'dataset_candle.parquet'
data_combined_path = Path('..') / 'data' / 'BTCUSDT' / 'processed' / 'dataset_combined.parquet'
data_orderbook_path = Path('..') / 'data' / 'BTCUSDT' / 'processed' / 'dataset_orderbook.parquet'
orderbook_features_path = Path('..') / 'data' / 'BTCUSDT' / 'processed' / 'orderbook_features.parquet'

print('reading', data_path.resolve())
print('reading', data_combined_path.resolve())
print('reading', data_orderbook_path.resolve())
print('reading', orderbook_features_path.resolve())

data = pd.read_parquet(data_path)
data_combined = pd.read_parquet(data_combined_path)
data_orderbook = pd.read_parquet(data_orderbook_path)
orderbook_features = pd.read_parquet(orderbook_features_path)

### Candle Data Exploration
Statistical overview, data types, distributions, and boxplot visualization of Candle features.

In [ ]:
print("Shape:", data.shape)
print("\nColumns:")
print(data.columns)

print("\nHead:")
print(data.head())
print("\nTail:")
print(data.tail())
print("\nMissing values per column:")
print(data.isna().sum())

print("\nNumeric summary:")
print(data.describe())

print("\nCategorical summary:")
print(data.describe(include="object"))

# Change Categorical data to numeric
obj_cols = ["quote_asset_volume", "taker_buy_base", "taker_buy_quote", "ignore"]
data[obj_cols] = data[obj_cols].apply(pd.to_numeric, errors="coerce")
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()

# Remove close_time from numeric columns for plotting
numeric_cols.remove('close_time')

print("\nInfo:")
print(data.info())
plt.figure(figsize=(12, 8))
data[numeric_cols].boxplot()
plt.title("Boxplot of Numeric Features")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Orderbook Data Exploration
Statistical overview and distribution analysis of orderbook microstructure features.

In [ ]:
print("Shape:", data_orderbook.shape)
print("\nColumns:")
print(data_orderbook.columns)

print("\nHead:")
print(data_orderbook.head())

print("\nMissing values per column:")
print(data_orderbook.isna().sum())

print("\nNumeric summary:")
print(data_orderbook.describe())

numeric_cols = data_orderbook.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('midprice')  # Remove midprice from numeric columns for plotting
print("\nInfo:")
print(data_orderbook.info())
plt.figure(figsize=(12, 8))
data_orderbook[numeric_cols].boxplot()
plt.title("Boxplot of Numeric Features")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Adding extra features

## Candle Model Pipeline
### Hyperparameter Tuning
Search for optimal XGBoost hyperparameters using candle OHLCV features with PredefinedSplit cross-validation.

In [ ]:
data = data.drop(columns=["ignore", "close_time"], errors="ignore")

X = data.drop(columns=['target'])
y_raw = data['target']                                                              
y= y_raw.map({-1: 0, 0: 1, 1: 2}).astype(int)  # Map -1 to 0, 0 to 1, and 1 to 2 for multiclass classification

gap = 5  # minutes
n = len(X)
train_end = int(n * 0.7)
val_end   = int(n * 0.85)

X_train_candle, y_train_candle = X.iloc[:train_end], y.iloc[:train_end]
X_val_candle, y_val_candle   = X.iloc[train_end+gap:val_end], y.iloc[train_end+gap:val_end]
X_test_candle, y_test_candle  = X.iloc[val_end+gap:], y.iloc[val_end+gap:]

class_counts = y_train_candle.value_counts().sort_index()
class_weights = (class_counts.sum() / class_counts).to_dict()
w_train = y_train_candle.map(class_weights).astype(float)


# Combine train + val into one "tuning" dataset
X_tune = np.vstack([X_train_candle.values, X_val_candle.values])
y_tune = np.concatenate([y_train_candle.values, y_val_candle.values])

# sample weights for val too (use train-derived weights or recompute on train+val)
class_counts = y_train_candle.value_counts().sort_index()
class_weights = (class_counts.sum() / class_counts).to_dict()
w_val_candle = y_val_candle.map(class_weights).astype(float)
w_tune_candle = np.concatenate([w_train.values, w_val_candle.values])

# tell sklearn: -1 means "always train", 0 means "validation fold"
test_fold = np.r_[np.full(len(X_train_candle), -1), np.zeros(len(X_val_candle))]
ps = PredefinedSplit(test_fold)

base = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    eval_metric="mlogloss",
    n_estimators=1000,
)

scorer = make_scorer(balanced_accuracy_score)

param_grid = {
    "max_depth": [2, 3, 4],
    "min_child_weight": [5, 10, 20],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
    "reg_lambda": [1.0, 5.0, 10.0],
    "gamma": [0.0, 0.1, 0.5],
    "learning_rate": [0.02, 0.05],
}

grid = GridSearchCV(
    base, param_grid=param_grid, scoring=scorer, cv=ps,
    n_jobs=-1, verbose=2, refit=True
)

grid.fit(X_tune, y_tune, sample_weight=w_tune_candle)

print("Best val score:", grid.best_score_)
print("Best params:", grid.best_params_)

best = grid.best_estimator_
pred = best.predict(X_test_candle)

print("Balanced accuracy:", balanced_accuracy_score(y_test_candle, pred))

### Train Final Candle Model with Early Stopping
Fit with early stopping on validation mlogloss, select optimal iterations, and refit on train+val data.

In [ ]:
# Training final candle model with early stopping
# Using best parameters found from grid search in previous cell
parameters = grid.best_params_

es = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    n_estimators=5000,
    eval_metric="mlogloss",
    early_stopping_rounds=200,
    random_state=42,
    n_jobs=-1,
    **parameters,
)

es.fit(
    X_train_candle, y_train_candle,
    sample_weight=w_train,
    eval_set=[(X_val_candle, y_val_candle)],
    verbose=False,
)

# Pick best iteration for initial decision weights
T = es.best_iteration

print("Best boosting round for initial weights:", T)

# Refit final model with optimal number of trees
final_candle = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    n_estimators=T,
    random_state=42,
    n_jobs=-1,
    **parameters,
)

# Train on train+val for final model
X_trval = pd.concat([X_train_candle, X_val_candle])
y_trval = pd.concat([y_train_candle, y_val_candle])

w_trval = pd.Series(y_trval).map(class_weights).astype(float).to_numpy()

final_candle.fit(X_trval, y_trval, sample_weight=w_trval, verbose=False)

print("Final candle model created with", final_candle.n_estimators, "trees")

### Visualize Candle Early Stopping Curve
Plot validation mlogloss across boosting rounds with early stopping iteration marked.

In [ ]:
# Visualize early stopping on candle model
results = es.evals_result()
val_loss = np.array(results["validation_0"]["mlogloss"])
best_it = es.best_iteration

plt.figure(figsize=(10, 5))
plt.plot(val_loss, label="val mlogloss")
plt.axvline(best_it, linestyle="--", color='red', label=f"early stopping (iter={best_it})")
plt.xlabel("Boosting round")
plt.ylabel("mlogloss")
plt.title("Candle Model - Early Stopping")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Candle Model Weight Optimization
Grid search for optimal class decision weights on validation set and evaluate on test set.

In [ ]:
# WEIGHT OPTIMIZATION: Find optimal class weights on validation set

def predict_weighted(proba, w):
    """Apply class weights to probabilities and make predictions."""
    proba_weighted = proba * np.array(w)[None, :]
    proba_weighted = proba_weighted / proba_weighted.sum(axis=1, keepdims=True)
    return proba_weighted.argmax(axis=1)

# Grid search for optimal weights on validation data
print("Searching for optimal class weights (down, flat, up)...")

best_w_candle, best_score_candle = None, -1

for w_down in [1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 2.0, 2.5, 3.0]:
    for w_flat in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
        for w_up in [1, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2, 3, 4]:
            proba_val = final_candle.predict_proba(X_val_candle)
            pred = predict_weighted(proba_val, (w_down, w_flat, w_up))
            s = f1_score(y_val_candle, pred, average="macro")
            if s > best_score_candle:
                best_score_candle, best_w_candle = s, (w_down, w_flat, w_up)

print(f"✓ Candle - Best w (down, flat, up): {best_w_candle}")
print(f"  Validation F1 score: {best_score_candle:.6f}")

# Apply best weights to test set
proba_test_candle = final_candle.predict_proba(X_test_candle)
pred_test_candle_opt = predict_weighted(proba_test_candle, best_w_candle)
test_acc_candle_opt = balanced_accuracy_score(y_test_candle, pred_test_candle_opt)
print(f"  Test balanced accuracy: {test_acc_candle_opt:.6f}\n")

### Candle Model Evaluation
Confusion matrix, classification report, and class balance analysis for candle model.

In [ ]:
print(confusion_matrix(y_test_candle, pred_test_candle_opt))
print(classification_report(y_test_candle, pred_test_candle_opt, digits=4, target_names=["down","flat","up"]))
print("training period", pd.Series(y_train_candle.value_counts(normalize=True).sort_index()))
print("test period", pd.Series(y_test_candle.value_counts(normalize=True).sort_index()))
print("validation period", pd.Series(y_val_candle.value_counts(normalize=True).sort_index()))

## Orderbook Model Pipeline
### Hyperparameter Tuning
Search for optimal XGBoost hyperparameters using orderbook microstructure features.

In [ ]:
X = data_orderbook.drop(columns=['target'])
y_raw = data_orderbook['target']                                                              
y= y_raw.map({-1: 0, 0: 1, 1: 2}).astype(int)  # Map -1 to 0, 0 to 1, and 1 to 2 for multiclass classification

gap = 5  # minutes
n = len(X)
train_end = int(n * 0.7)
val_end   = int(n * 0.85)

X_train_orderbook, y_train_orderbook = X.iloc[:train_end], y.iloc[:train_end]
X_val_orderbook, y_val_orderbook   = X.iloc[train_end+gap:val_end], y.iloc[train_end+gap:val_end]
X_test_orderbook, y_test_orderbook  = X.iloc[val_end+gap:], y.iloc[val_end+gap:]

class_counts = y_train_orderbook.value_counts().sort_index()
class_weights = (class_counts.sum() / class_counts).to_dict()
w_train = y_train_orderbook.map(class_weights).astype(float)


# Combine train + val into one "tuning" dataset
X_tune_orderbook = np.vstack([X_train_orderbook.values, X_val_orderbook.values])
y_tune_orderbook = np.concatenate([y_train_orderbook.values, y_val_orderbook.values])

# sample weights for val too (use train-derived weights or recompute on train+val)
class_counts = y_train_orderbook.value_counts().sort_index()
class_weights = (class_counts.sum() / class_counts).to_dict()
w_val_orderbook = y_val_orderbook.map(class_weights).astype(float)
w_tune_orderbook = np.concatenate([w_train.values, w_val_orderbook.values])

# tell sklearn: -1 means "always train", 0 means "validation fold"
test_fold = np.r_[np.full(len(X_train_orderbook), -1), np.zeros(len(X_val_orderbook))]
ps = PredefinedSplit(test_fold)

base = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    eval_metric="mlogloss",
    n_estimators=800,
)

scorer = make_scorer(balanced_accuracy_score)

param_grid = {
    "max_depth": [2, 3, 4],
    "min_child_weight": [5, 10, 20],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
    "reg_lambda": [1.0, 5.0, 10.0],
    "gamma": [0.0, 0.1, 0.2],
    "learning_rate": [0.02, 0.05],
}

grid = GridSearchCV(
    base, param_grid=param_grid, scoring=scorer, cv=ps,
    n_jobs=-1, verbose=2, refit=True
)

grid.fit(X_tune_orderbook, y_tune_orderbook, sample_weight=w_tune_orderbook)

print("Best val score:", grid.best_score_)
print("Best params:", grid.best_params_)

best = grid.best_estimator_
pred = best.predict(X_test_orderbook)

print("Balanced accuracy:", balanced_accuracy_score(y_test_orderbook, pred))
# Best val score: 0.40920175692524313
# Best params: {'colsample_bytree': 0.7, 'gamma': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'min_child_weight': 20, 'reg_lambda': 1.0, 'subsample': 1.0}
# Balanced accuracy: 0.44006290892987715

### Train Final Orderbook Model with Early Stopping
Fit with early stopping on validation mlogloss, select optimal iterations, and refit on train+val data.

In [ ]:
parameters = grid.best_params_
# 1) Fit with early stopping on a smooth metric
es = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    n_estimators=5000,
    eval_metric="mlogloss",
    early_stopping_rounds=200,
    random_state=42,
    n_jobs=-1,
    # plus your tuned params:
    **parameters,  # <-- dict with max_depth, subsample, etc
)

es.fit(
    X_train_orderbook, y_train_orderbook,
    sample_weight=w_train,
    eval_set=[(X_val_orderbook, y_val_orderbook)],
    verbose=False,
)

# 2) Pick best iteration for *your* metric (weighted decision rule)
T = es.best_iteration

print("Best boosting round for weighted balanced acc:", T)

# 3) Refit final model (no early stopping), using that number of trees
final_orderbook = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    n_estimators=T,
    random_state=42,
    n_jobs=-1,
    **parameters,
)

# optional: train on train+val now that best_t is chosen
X_trval = np.concatenate([X_train_orderbook.values, X_val_orderbook.values], axis=0)
y_trval = np.concatenate([y_train_orderbook.values, y_val_orderbook.values], axis=0)

final_orderbook.fit(X_trval, y_trval, verbose=False)

# test with the same decision rule
proba_test = final_orderbook.predict_proba(X_test_orderbook)
pred_test = (proba_test).argmax(axis=1)
# test with the same decision rule
print("Final candle model created with", final_orderbook.n_estimators, "trees")
print("Test weighted balanced acc:", balanced_accuracy_score(y_test_orderbook, pred_test))

### Orderbook Model Weight Optimization
Grid search for optimal class decision weights on validation set and evaluate on test set.

In [ ]:
# ============================================================================
# WEIGHT OPTIMIZATION: Find optimal class weights on validation set
# ============================================================================

def predict_weighted(proba, w):
    """Apply class weights to probabilities and make predictions."""
    proba_weighted = proba * np.array(w)[None, :]
    proba_weighted = proba_weighted / proba_weighted.sum(axis=1, keepdims=True)
    return proba_weighted.argmax(axis=1)

# Grid search for optimal weights on validation data
print("Searching for optimal class weights (down, flat, up)...")
print("This may take a minute...\n")

best_w_orderbook, best_score_orderbook = None, -1

for w_down in [1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 2.0, 2.5, 3.0]:
    for w_flat in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
        for w_up in [1, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2, 3, 4]:
            proba_val = final_orderbook.predict_proba(X_val_orderbook)
            pred = predict_weighted(proba_val, (w_down, w_flat, w_up))
            s = f1_score(y_val_orderbook, pred, average="macro")
            if s > best_score_orderbook:
                best_score_orderbook, best_w_orderbook = s, (w_down, w_flat, w_up)

print(f"✓ Orderbook - Best w (down, flat, up): {best_w_orderbook}")
print(f"  Validation F1 score: {best_score_orderbook:.6f}")

# Apply best weights to test set
proba_test_orderbook = final_orderbook.predict_proba(X_test_orderbook)
pred_test_orderbook_opt = predict_weighted(proba_test_orderbook, best_w_orderbook)
test_acc_orderbook_opt = balanced_accuracy_score(y_test_orderbook, pred_test_orderbook_opt)
print(f"  Test balanced accuracy (optimized): {test_acc_orderbook_opt:.6f}\n")

### Orderbook Model Evaluation
Confusion matrix, classification report, and class balance analysis for orderbook model.

In [ ]:
print(confusion_matrix(y_test_orderbook, pred_test_orderbook_opt))
print(classification_report(y_test_orderbook, pred_test_orderbook_opt, digits=4, target_names=["down","flat","up"]))
print("training period", pd.Series(y_train_orderbook.value_counts(normalize=True).sort_index()))
print("test period", pd.Series(y_test_orderbook.value_counts(normalize=True).sort_index()))
print("validation period", pd.Series(y_val_orderbook.value_counts(normalize=True).sort_index()))

## Ensemble: Candle + Orderbook Blending
Find optimal alpha weight to blend candle and orderbook model probabilities for combined predictions.

In [ ]:
assert X_val_candle.index.equals(X_val_orderbook.index)
assert X_test_candle.index.equals(X_test_orderbook.index)
assert y_val_candle.index.equals(y_val_orderbook.index)
assert y_test_candle.index.equals(y_test_orderbook.index)

def weight_probs(proba, w):
    w = np.array(w)
    proba_w = proba * w[None, :]
    proba_w = proba_w / proba_w.sum(axis=1, keepdims=True)
    return proba_w

# Get validation probabilities and apply optimized decision weights per model
p_val_c = weight_probs(final_candle.predict_proba(X_val_candle), best_w_candle)
p_val_o = weight_probs(final_orderbook.predict_proba(X_val_orderbook), best_w_orderbook)

best_alpha, best_score = None, -1

for alpha in np.linspace(0, 1, 21):  # 0.00 ... 1.00
    p_mix = alpha * p_val_c + (1 - alpha) * p_val_o
    pred = p_mix.argmax(axis=1)
    score = balanced_accuracy_score(y_val_candle, pred)
    if score > best_score:
        best_score, best_alpha = score, alpha

# Compute test score using the same weighted decision rules
p_test_c = weight_probs(final_candle.predict_proba(X_test_candle), best_w_candle)
p_test_o = weight_probs(final_orderbook.predict_proba(X_test_orderbook), best_w_orderbook)
p_test_mix = best_alpha * p_test_c + (1 - best_alpha) * p_test_o
pred_test = p_test_mix.argmax(axis=1)
test_score = balanced_accuracy_score(y_test_candle, pred_test)

print(f"Best alpha: {best_alpha}  Val balanced acc: {best_score:.6f}  Test balanced acc: {test_score:.6f}")


## Combined Model Pipeline
### Combined Data Exploration
Statistical overview and distribution analysis of combined candle + orderbook features.

Creating a model with the combined data set of orderbook and candle based data. First exploring combined data,

### Identify Tail-Heavy Features
Analyze feature distributions using p99/p50 ratio to identify candidates for log transformation.

In [ ]:
print("Shape:", data_combined.shape)
print("\nColumns:")
print(data_combined.columns)

print("\nHead:")
print(data_combined.head())
print("\nTail:")
print(data_combined.tail())
print("\nMissing values per column:")
print(data_combined.isna().sum())

print("\nNumeric summary:")
print(data_combined.describe())

# Change Categorical data to numeric
obj_cols = ["quote_asset_volume", "taker_buy_base", "taker_buy_quote", "ignore"]
data_combined[obj_cols] = data_combined[obj_cols].apply(pd.to_numeric, errors="coerce")
numeric_cols = data_combined.select_dtypes(include=[np.number]).columns.tolist()

#drop duplicate and stationary features
data_combined = data_combined.drop(columns=["close_time", "ignore"], errors="ignore")
numeric_cols.remove('quote_asset_volume')

numeric_cols.remove('taker_buy_quote') 

print("\nInfo:")
print(data_combined.info())
plt.figure(figsize=(12, 8))
data_combined.boxplot()
plt.title("Boxplot of Numeric Features")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Apply Log Transform
Apply log1p transformation to tail-heavy volume and flow features for distribution normalization.

In [ ]:
cols = [c for c in data_combined.columns if c != "target"]
q = data_combined[cols].quantile([0.5, 0.99])
tail = (q.loc[0.99] / q.loc[0.5].replace(0, np.nan)).sort_values(ascending=False)
print(tail.head(10))


Using specifically Log1p here. 
What is does: log1p(x) = log(1 + x).

- For small values, it behaves almost linear: log1p(x) ≈ x
- For large values, it compresses hard: going from 10 → 1000 becomes a much smaller jump than in raw space

So it turns “rare huge spikes” into “moderate bumps,” which makes patterns easier for models to learn.
It’s also safe for zeros (unlike plain log(x)), which is why log1p is standard for volumes/flows/counts.

In [ ]:
log_cols = ["taker_buy_base", "taker_buy_quote", "volume", "quote_asset_volume", "num_trades"]
for c in log_cols:
    data_combined[c] = np.log1p(data_combined[c].clip(lower=0))

### Normalize Volatility Metrics
Convert Bollinger Band width and ATR to percentage of price for price-scale independence.

In [ ]:
data_combined["ATR_pct"] = data_combined["ATR"] / data_combined["close"] 
data_combined["BB_width_pct"] = data_combined["BB_width"] / data_combined["close"]

### Normalize Bid-Ask Spread
Convert spread to basis points (bps) normalized by midprice for scale independence.

In [ ]:
data_combined["spread_bps"] = data_combined["bid_ask_spread"].abs() / data_combined["midprice"] * 1e4

### Compute Order Book Depth Metrics
Calculate total depth and depth imbalance as relative measures of liquidity and flow imbalance.

In [ ]:
data_combined["depth_sum"] = data_combined["bid_depth"] + data_combined["ask_depth"] # Total depth on both sides of the order book
data_combined["depth_imb"] = (data_combined["bid_depth"] - data_combined["ask_depth"]) / (data_combined["depth_sum"] + 1e-12) # Depth imbalance

### Verify Feature Engineering
Confirm all raw features have been converted and visualize final feature distributions.

In [ ]:
data_combined = data_combined.drop(columns=["ATR", "BB_width","BB_upper", "BB_lower","bid_ask_spread", "midprice", "bid_depth", "ask_depth", "open", "high", "low", "close"])

Print a boxplot of the current data again.

In [ ]:
print("\nInfo:")
print(data_combined.info())
plt.figure(figsize=(12, 8))
data_combined.boxplot()
plt.title("Boxplot of Numeric Features")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Combined Hyperparameter Tuning
Grid search for optimal XGBoost hyperparameters using combined features with balanced accuracy scoring.

In [ ]:
X_combined = data_combined.drop(columns=['target'])
y_raw_combined = data_combined['target']                                                              
y_combined = y_raw_combined.map({-1: 0, 0: 1, 1: 2}).astype(int)  # Map -1 to 0, 0 to 1, and 1 to 2 for multiclass classification

gap = 5  # minutes
n = len(X_combined)
train_end = int(n * 0.7)
val_end   = int(n * 0.85)

X_train_combined, y_train_combined = X_combined.iloc[:train_end], y_combined.iloc[:train_end]
X_val_combined, y_val_combined   = X_combined.iloc[train_end+gap:val_end], y_combined.iloc[train_end+gap:val_end]
X_test_combined, y_test_combined  = X_combined.iloc[val_end+gap:], y_combined.iloc[val_end+gap:]

class_counts = y_train_combined.value_counts().sort_index()
class_weights = (class_counts.sum() / class_counts).to_dict()
w_train_combined = y_train_combined.map(class_weights).astype(float)


# Combine train + val into one "tuning" dataset
X_tune_combined = np.vstack([X_train_combined.values, X_val_combined.values])
y_tune_combined = np.concatenate([y_train_combined.values, y_val_combined.values])

# sample weights for val too (use train-derived weights or recompute on train+val)
class_counts = y_train_combined.value_counts().sort_index()
class_weights = (class_counts.sum() / class_counts).to_dict()
w_val_combined = y_val_combined.map(class_weights).astype(float)
w_tune_combined = np.concatenate([w_train_combined.values, w_val_combined.values])

# tell sklearn: -1 means "always train", 0 means "validation fold"
test_fold = np.r_[np.full(len(X_train_combined), -1), np.zeros(len(X_val_combined))]
ps = PredefinedSplit(test_fold)

base = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    eval_metric="mlogloss",
    n_estimators=800,
)

scorer = make_scorer(balanced_accuracy_score)

param_grid = {
    "max_depth": [2, 3, 4],
    "min_child_weight": [5, 10, 20],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
    "reg_lambda": [1.0, 5.0, 10.0],
    "gamma": [0.0, 0.1, 0.2],
    "learning_rate": [0.02, 0.05],
}

grid = GridSearchCV(
    base, param_grid=param_grid, scoring=scorer, cv=ps,
    n_jobs=-1, verbose=2, refit=True
)

grid.fit(X_tune_combined, y_tune_combined, sample_weight=w_tune_combined)

print("Best val score:", grid.best_score_)
print("Best params:", grid.best_params_)

best = grid.best_estimator_
pred = best.predict(X_test_combined)

print("Balanced accuracy:", balanced_accuracy_score(y_test_combined, pred))

# Best val score: 0.42164786640819835
# Best params: {'colsample_bytree': 0.7, 'gamma': 0.2, 'learning_rate': 0.05, 'max_depth': 3, 'min_child_weight': 20, 'reg_lambda': 5.0, 'subsample': 1.0}
# Balanced accuracy: 0.455582011581993


### Train Final Combined Model with Early Stopping
Fit with early stopping on validation mlogloss, select optimal iterations, and refit on train+val data.

In [ ]:
parameters = grid.best_params_
# 1) Fit with early stopping on a smooth metric
es = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    n_estimators=5000,
    eval_metric="mlogloss",
    early_stopping_rounds=200,
    random_state=42,
    n_jobs=-1,
    # plus your tuned params:
    **parameters,  # dictionary with max_depth, subsample, etc
)

es.fit(
    X_train_combined, y_train_combined,
    sample_weight=w_train_combined,
    eval_set=[(X_val_combined, y_val_combined)],
    verbose=False,
)

# 2) Pick best iteration for *your* metric (weighted decision rule)
T = es.best_iteration

# Refit final model (no early stopping), using that number of trees
final_combined = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    n_estimators=T,
    random_state=42,
    n_jobs=-1,
    **parameters,
)

# optional: train on train+val now that best_t is chosen
X_trval = np.concatenate([X_train_combined.values, X_val_combined.values], axis=0)
y_trval = np.concatenate([y_train_combined.values, y_val_combined.values], axis=0)

final_combined.fit(X_trval, y_trval, verbose=False)

# test with the same decision rule
proba_test = final_combined.predict_proba(X_test_combined)
pred_test = (proba_test).argmax(axis=1)
print("Test weighted balanced acc:", balanced_accuracy_score(y_test_combined, pred_test))

### Combined Model Weight Optimization
Grid search for optimal class decision weights on validation set and evaluate on test set.

In [ ]:
# ============================================================================
# WEIGHT OPTIMIZATION: Find optimal class weights on validation set
# ============================================================================

def predict_weighted(proba, w):
    """Apply class weights to probabilities and make predictions."""
    proba_weighted = proba * np.array(w)[None, :]
    proba_weighted = proba_weighted / proba_weighted.sum(axis=1, keepdims=True)
    return proba_weighted.argmax(axis=1)

# Grid search for optimal weights on validation data
print("Searching for optimal class weights (down, flat, up)...")
print("This may take a minute...\n")

best_w_combined, best_score_combined = None, -1

for w_down in [1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 2.0, 2.5, 3.0]:
    for w_flat in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
        for w_up in [1, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2, 3, 4]:
            proba_val = final_combined.predict_proba(X_val_combined)
            pred = predict_weighted(proba_val, (w_down, w_flat, w_up))
            s = f1_score(y_val_combined, pred, average="macro")
            if s > best_score_combined:
                best_score_combined, best_w_combined = s, (w_down, w_flat, w_up)

print(f"✓ Combined - Best w (down, flat, up): {best_w_combined}")
print(f"  Validation F1 score: {best_score_combined:.6f}")

# Apply best weights to test set
proba_test_combined = final_combined.predict_proba(X_test_combined)
pred_test_combined_opt = predict_weighted(proba_test_combined, best_w_combined)
test_acc_combined_opt = balanced_accuracy_score(y_test_combined, pred_test_combined_opt)
print(f"  Test balanced accuracy (optimized): {test_acc_combined_opt:.6f}\n")

### Combined Model Evaluation
Confusion matrix, classification report, and class balance analysis for combined model.

In [ ]:
print(confusion_matrix(y_test_combined, pred_test_combined_opt))
print(classification_report(y_test_combined, pred_test_combined_opt, digits=4, target_names=["down","flat","up"]))
print("training period", pd.Series(y_train_combined.value_counts(normalize=True).sort_index()))
print("test period", pd.Series(y_test_combined.value_counts(normalize=True).sort_index()))
print("validation period", pd.Series(y_val_combined.value_counts(normalize=True).sort_index()))

### Visualize Combined Early Stopping Curve
Plot validation mlogloss across boosting rounds with early stopping iteration marked.

In [ ]:
results = es.evals_result()

val_acc  = np.array(results["validation_0"]["mlogloss"])

best_it = es.best_iteration  # chosen by early stopping

plt.figure(figsize=(10, 5))
plt.plot(val_acc, label="val mlogloss")
plt.axvline(best_it, linestyle="--", label=f"best iter={best_it}")
plt.xlabel("Boosting round")
plt.ylabel("mlogloss")
plt.title("Combined Model - Early Stopping")
plt.legend()
plt.show()

## Final Model Comparison Summary

In [ ]:
# MODEL COMPARISON SUMMARY

# Collect all model results using calculated values from weight optimization
models_summary = pd.DataFrame({
    'Model': ['Candle', 'Orderbook', 'Combined', 'Ensemble (Candle+Orderbook)'],
    'Features': ['OHLCV', 'Orderbook Microstructure', 'Combined (OHLCV + Orderbook)', 'Linear combination (Candle+Orderbook)'],
    'Best Iteration': [
        final_candle.n_estimators, 
        final_orderbook.n_estimators, 
        final_combined.n_estimators,
        np.nan
    ],
    'Test Balanced Accuracy': [
        test_acc_candle_opt,
        test_acc_orderbook_opt,
        test_acc_combined_opt,
        test_score,
    ],
    'Optimal Decision Weights (D, F, U)': [
        str(best_w_candle),
        str(best_w_orderbook),
        str(best_w_combined),
        f"alpha={best_alpha:.3f}",
    ]
})

# Display summary
print("\n" + "="*100)
print("MODEL COMPARISON SUMMARY".center(100))
print("="*100)
print(models_summary.to_string(index=False))
print("="*100)

# Key insights
print(f"  • Best performing model: {models_summary.loc[models_summary['Test Balanced Accuracy'].idxmax(), 'Model']}")
print(f"    Test Balanced Accuracy: {models_summary['Test Balanced Accuracy'].max():.4f}")
print(f"\n  • Candle model: Quick OHLCV-based predictions")
print(f"  • Orderbook model: Market microstructure signals")
print(f"  • Combined model: Integrated multi-source features")
print("\n" + "="*100)


## Feature Importance Analysis

Analyze the magnitude of coefficients for each class to understand feature importance.

In [ ]:
# FEATURE IMPORTANCE ANALYSIS FOR XGBOOST MODELS
importances = {}
def plot_xgb_feature_importance(model, feature_names, dataset_name, top_n=15):
    """
    Plot feature importance for XGBoost model using gain importance.
    """
    booster = model.get_booster()
    importance_dict = booster.get_score(importance_type='gain')
    # Map feature indices to names if needed
    if all(k.startswith('f') for k in importance_dict.keys()):
        fmap = {f'f{i}': name for i, name in enumerate(feature_names)}
        importance_dict = {fmap.get(k, k): v for k, v in importance_dict.items()}
    
    importance_df = pd.DataFrame({
        'feature': list(importance_dict.keys()),
        'importance': list(importance_dict.values())
    }).sort_values('importance', ascending=False)
    
    top_features = importance_df.head(top_n)
    plt.figure(figsize=(10, 6))
    plt.barh(range(len(top_features)), top_features['importance'].values)
    plt.yticks(range(len(top_features)), top_features['feature'].values)
    plt.xlabel('Gain Importance', fontsize=11)
    plt.ylabel('Feature', fontsize=11)
    plt.title(f'Top {top_n} Features - {dataset_name} (XGBoost)', fontsize=12, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    return importance_df

# Plot for each dataset (Candle, Orderbook, Combined)
print("\n" + "="*70)
print("XGBOOST FEATURE IMPORTANCE ANALYSIS")
print("="*70)

# Candle
candle_importance_xgb = plot_xgb_feature_importance(final_candle, X_train_candle.columns, 'Candle')
importances['candle'] = candle_importance_xgb

# Orderbook
orderbook_importance_xgb = plot_xgb_feature_importance(final_orderbook, X_train_orderbook.columns, 'Orderbook')
importances['orderbook'] = orderbook_importance_xgb

# Combined
combined_importance_xgb = plot_xgb_feature_importance(final_combined, X_train_combined.columns, 'Combined', top_n=20)
importances['combined'] = combined_importance_xgb

## Regularization Analysis

Plot validation performance across different C values for each dataset.

In [ ]:
# REGULARIZATION ANALYSIS: LAMBDA (L2) TUNING FOR XGBOOST

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Each dataset: Candle, Orderbook, Combined
datasets = [
    ('Candle', X_train_candle, y_train_candle, X_val_candle, y_val_candle),
    ('Orderbook', X_train_orderbook, y_train_orderbook, X_val_orderbook, y_val_orderbook),
    ('Combined', X_train_combined, y_train_combined, X_val_combined, y_val_combined)
    ]

lambdas = [0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10, 20, 50, 100]
for idx, (name, Xtr, ytr, Xval, yval) in enumerate(datasets):
    val_bal_accs = []
    val_f1s = []
    for reg_lambda in lambdas:
        model = xgb.XGBClassifier(
            objective="multi:softprob",
            num_class=3,
            tree_method="hist",
            n_estimators=200,
            reg_lambda=reg_lambda,
            random_state=42,
            n_jobs=-1,
            eval_metric="mlogloss",
            verbosity=0,
        )
        model.fit(Xtr, ytr, eval_set=[(Xval, yval)], verbose=False)
        pred = model.predict(Xval)
        val_bal_accs.append(balanced_accuracy_score(yval, pred))
        val_f1s.append(f1_score(yval, pred, average="macro"))

    ax = axes[idx]
    ax.plot(lambdas, val_bal_accs, 'o-', label='Balanced Acc', linewidth=2, markersize=8)
    ax.plot(lambdas, val_f1s, 's-', label='Macro F1', linewidth=2, markersize=8)
    best_idx = int(np.argmax(val_bal_accs))
    best_lambda = lambdas[best_idx]
    ax.axvline(best_lambda, color='red', linestyle='--', alpha=0.7, label=f'Best λ={best_lambda}')
    ax.set_xscale('log')
    ax.set_xlabel('λ (L2 Regularization)', fontsize=11)
    ax.set_ylabel('Validation Score', fontsize=11)
    ax.set_title(f'{name} Dataset', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('L2 Regularization Tuning (XGBoost)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Analysis Confusion Matrix

In [ ]:
# CONFUSION MATRIX ANALYSIS FOR XGBOOST MODELS

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Each dataset: Candle, Orderbook, Combined
datasets_cm = [
    ('Candle', y_test_candle, final_candle.predict(X_test_candle)),
    ('Orderbook', y_test_orderbook, final_orderbook.predict(X_test_orderbook)),
    ('Combined', y_test_combined, final_combined.predict(X_test_combined)),
]

for idx, (name, y_true, y_pred) in enumerate(datasets_cm):
    ax = axes[idx]
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues', 
                xticklabels=['Down', 'Flat', 'Up'],
                yticklabels=['Down', 'Flat', 'Up'],
                ax=ax, cbar_kws={'label': 'Proportion'})
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_ylabel('True Label', fontsize=11)
    ax.set_title(f'{name} Dataset', fontsize=12, fontweight='bold')

plt.suptitle('Normalized Confusion Matrices (XGBoost)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()